# Projeto Churn Predictor MLOps

Notebook acadêmico End-to-End: entendimento do problema, EDA, preparação, comparação de modelos, MLflow e exportação do Champion para a API.

## 1. Instalação e importações
Execute a instalação apenas no Google Colab.

In [ ]:
# !pip install -q pandas scikit-learn xgboost imbalanced-learn mlflow joblib seaborn
import json
from pathlib import Path
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

## 2. Carregamento e entendimento dos dados
A variável alvo é `Churn`: `Yes` significa que o cliente cancelou e `No` que permaneceu.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/SaeidRostami/Customer_Churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_URL)
print('Formato:', df.shape)
print('Vazios escondidos em TotalCharges:', pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum())
display(df.head())
display(df['Churn'].value_counts())
display((df['Churn'].value_counts(normalize=True) * 100).round(2))

## 3. Limpeza e separação
O identificador não deve ser usado para prever. `TotalCharges` é convertido em número; seus 11 valores vazios serão imputados pela mediana dentro da pipeline.

In [ ]:
dados = df.copy()
dados['TotalCharges'] = pd.to_numeric(dados['TotalCharges'], errors='coerce')
X = dados.drop(columns=['customerID', 'Churn'])
y = dados['Churn'].map({'No': 0, 'Yes': 1})
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Treino:', X_treino.shape, 'Teste:', X_teste.shape)

## 4. Insights de negócio

In [ ]:
eda = dados.assign(Churn_num=y)
for coluna in ['Contract', 'InternetService', 'PaymentMethod']:
    taxas = (eda.groupby(coluna)['Churn_num'].mean() * 100).sort_values(ascending=False)
    print(f'\nTaxa de churn por {coluna}:')
    display(taxas.round(2))
print('Tenure médio:', eda.groupby('Churn_num')['tenure'].mean().round(2).to_dict())
print('MonthlyCharges médio:', eda.groupby('Churn_num')['MonthlyCharges'].mean().round(2).to_dict())

## 5. Pipeline de preparação
A pipeline evita vazamento de dados e garante que o mesmo tratamento seja aplicado no treinamento e na API.

In [ ]:
colunas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
colunas_categoricas = [c for c in X.columns if c not in colunas_numericas]
pipeline_numerica = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escala', StandardScaler())
])
pipeline_categorica = Pipeline([
    ('imputador', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessador = ColumnTransformer([
    ('numericas', pipeline_numerica, colunas_numericas),
    ('categoricas', pipeline_categorica, colunas_categoricas)
])

## 6. Treinamento e comparação
Recall recebe atenção especial porque perder um cliente sem identificá-lo tende a ser mais caro que uma abordagem preventiva desnecessária.

In [ ]:
modelos = {
    'Regressão Logística': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=250, max_depth=2, learning_rate=0.03, subsample=0.8, colsample_bytree=0.9, scale_pos_weight=2.55, eval_metric='logloss', random_state=42, n_jobs=-1)
}
resultados = []
pipelines = {}
for nome, estimador in modelos.items():
    pipeline = Pipeline([('preprocessamento', preprocessador), ('modelo', estimador)])
    pipeline.fit(X_treino, y_treino)
    previsoes = pipeline.predict(X_teste)
    probabilidades = pipeline.predict_proba(X_teste)[:, 1]
    resultados.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_teste, previsoes),
        'Precision': precision_score(y_teste, previsoes),
        'Recall': recall_score(y_teste, previsoes),
        'F1-score': f1_score(y_teste, previsoes),
        'ROC AUC': roc_auc_score(y_teste, probabilidades)
    })
    pipelines[nome] = pipeline
tabela_resultados = pd.DataFrame(resultados).set_index('Modelo').round(4)
display(tabela_resultados)

## 7. MLflow e registro do experimento

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('churn-prediction')
for resultado in resultados:
    nome = resultado['Modelo']
    with mlflow.start_run(run_name=nome.lower().replace(' ', '-')):
        mlflow.log_param('model_type', nome)
        mlflow.log_metrics({
            'accuracy_test': resultado['Acurácia'],
            'precision_test': resultado['Precision'],
            'recall_test': resultado['Recall'],
            'f1_test': resultado['F1-score'],
            'roc_auc_test': resultado['ROC AUC']
        })
        mlflow.sklearn.log_model(pipelines[nome], name='model')

## 8. Exportação do Champion
O artefato contém pré-processamento e XGBoost na mesma pipeline, evitando inconsistência entre treino e produção.

In [ ]:
champion = pipelines['XGBoost']
pasta_modelos = Path('../models')
pasta_modelos.mkdir(parents=True, exist_ok=True)
joblib.dump(champion, pasta_modelos / 'churn_model.joblib')
print('Champion salvo em:', pasta_modelos / 'churn_model.joblib')